# Laboratorio 07 · Sesión 2
**Versión estudiante con TODO.** Curso CIIN1021P. Datos sintéticos para fines académicos.

Ejecuta las celdas **en orden**. En los bloques TODO, lee los comentarios numerados (empiezan con `#`) y reemplaza cada `____` por lo que indican. No ejecutes una celda con `____` pendientes. Los comentarios te dicen **qué lograr**; la palabra exacta la eliges tú con la *Referencia rápida*.

Entorno: Colab CPU, PySpark 4.0.3, Java 17, Spark local[2]. Esta sesión no depende de la anterior.

**Duración: 90 minutos.** Arranque 10 · Ejercicio 03 25 · Ejercicio 04 25 · Ejercicio 05 25 · cierre 5. El reto integrador es opcional y queda fuera de los 90 minutos.

## Paso 0 · Preparar Colab
Sube este archivo en *Archivo > Subir cuaderno*. Usa un entorno CPU. Si se reinicia la máquina, vuelve a ejecutar este paso completo.

In [ ]:
# Paso 0. Instalar la versión utilizada en esta práctica.
%pip install -q pyspark==4.0.3

In [ ]:
# Paso 0. Verificar Java en la máquina de Colab.
import os, sys, subprocess, shutil
from pathlib import Path

java17 = Path('/usr/lib/jvm/java-17-openjdk-amd64')
if not java17.exists():
    # Esta instalación se realiza en la máquina temporal de Colab.
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq',
                    'openjdk-17-jre-headless'], check=True)
os.environ['JAVA_HOME'] = str(java17)
os.environ['PATH'] = str(java17 / 'bin') + os.pathsep + os.environ['PATH']
os.environ['PYSPARK_PYTHON'] = sys.executable
print('Python:', sys.version.split()[0])
print(subprocess.check_output(['java', '-version'],
      stderr=subprocess.STDOUT, text=True))

In [ ]:
# Paso 0. Iniciar Spark Classic en una sola máquina con dos hilos.
from pyspark.sql import SparkSession, functions as F, types as T
spark = (SparkSession.builder.appName('UPN_Tema07')
         .master('local[2]')
         .config('spark.sql.shuffle.partitions', '2')
         .config('spark.sql.session.timeZone', 'UTC')
         .getOrCreate())
spark.sparkContext.setLogLevel('ERROR')
print('Spark:', spark.version)
print('Modo:', spark.sparkContext.master)
assert spark.range(5).count() == 5
print('Prueba de arranque correcta')

Spark: 4.0.3
Modo: local[2]
Prueba de arranque correcta


In [ ]:
# Paso 0. Subir únicamente datos_tema07.zip desde tu computadora.
from google.colab import files
from pathlib import Path
import zipfile

subidos = files.upload()
nombre = next((n for n in subidos if n.endswith('.zip')), None)
if nombre is None:
    raise ValueError('Sube el archivo datos_tema07.zip')
BASE = Path('/content/datos_tema07')
BASE.mkdir(exist_ok=True)
esperados = {'movitrack_control.csv', 'playas.csv', 'jornadas.csv',
             'residuos.csv', 'sensores.csv'}
with zipfile.ZipFile(nombre) as z:
    assert esperados.issubset(set(z.namelist())), 'ZIP de datos incorrecto'
    for archivo in esperados:
        (BASE / archivo).write_bytes(z.read(archivo))
print('Datos disponibles:', sorted(p.name for p in BASE.glob('*.csv')))

In [ ]:
# Paso 0. Función de ayuda para comprobar tus resultados.
# Compara lo que obtuviste con el valor esperado y muestra ✔ o ✘ (no necesitas entenderla).
ESPERADO = {
    'recibidos': 1809, 'duplicados': 5,
    'peso_nulo': 1, 'peso_no_positivo': 1, 'tipo_nulo': 1, 'tipo_fuera_lista': 0, 'huerfanos': 1,
    'motivos': {'Jornada no existente': 1, 'Peso ausente': 1, 'Peso no positivo': 1, 'Tipo no reconocido': 1},
    'conteos': (1809, 5, 1800, 4), 'aceptacion': 99.78,
    'por_region': {'Centro': (600, 11998.0, 11.998), 'Norte': (600, 5998.0, 5.998), 'Sur': (600, 17998.0, 17.998)},
    'estadistica': {'Centro': (200, 398.08, 51.38), 'Norte': (200, 749.9, 213.39), 'Sur': (200, 295.63, 26.86)},
    'impacto': {'Alto Impacto': 303, 'Bajo Impacto': 297},
    'parquet': 1800, 'csv_filas': 3, 'toneladas_total': 35.994,
}
def verificar(nombre, obtenido):
    if obtenido == ESPERADO[nombre]:
        print('✔', nombre, 'correcto')
    else:
        print('✘', nombre, 'no coincide. Obtuviste:', obtenido)
        print('   Revisa los comentarios numerados del TODO y vuelve a ejecutar.')

## Referencia rápida (consúltala en los TODO)
| Quiero… | Opciones en PySpark (hay distractores) |
|---|---|
| Preguntar por valores vacíos | `F.col('c').isNull()` · `F.col('c').isNotNull()` · `F.col('c').isin(...)` |
| Negar una condición | `~condición` |
| Comparar números | `>` · `>=` · `<` · `<=` · `==` |
| Calcular por grupo | `F.count` · `F.sum` · `F.avg` · `F.stddev_samp` · `F.stddev_pop` · `F.max` |
| Unir tablas | `join(otra, 'clave', 'inner')` · `'left'` · `'left_anti'` |
| Sacar un resumen **pequeño** a pandas | `toPandas()` · `collect()` · `show()` |

*Si una opción no te convence, léela en voz alta: ¿qué pregunta responde?*

## Ejercicio 03 · Depurar los datos de Ocean Clean Up Peru (25 min)

**Qué lograremos.** La ONG recibe pesajes de residuos de sus jornadas. Antes de calcular cualquier indicador debemos separar los **duplicados exactos** y los **registros que incumplen reglas**, sin borrar ni modificar los archivos originales.

**Por qué importa.** Un dato sucio no produce error: produce un indicador equivocado sin avisar. Aquí lo comprobarás con un caso real del ejercicio: un registro defectuoso que un filtro aparentemente correcto *no detecta*.

**Datos.** 6 playas ficticias, 600 jornadas y 1800 pesajes válidos. Se añadieron 5 duplicados exactos y 4 registros defectuosos. Una fila de `residuos` es **un pesaje** de un tipo de residuo en una jornada.

```
playas (6)  ──PlayaID──▶  jornadas (600)  ──JornadaID──▶  residuos (1809 filas)
```

In [ ]:
# Paso 1. Leer las tres tablas con tipos explícitos (mismo patrón del Ejercicio 01).
def leer(nombre, esquema):
    return (spark.read.option('header', True)   # la primera línea trae los nombres de columna
            .schema(esquema)                    # los tipos los fijamos nosotros
            .csv(str(BASE / nombre)))

playas = leer('playas.csv', 'PlayaID INT, Nombre STRING, Region STRING')
jornadas = leer('jornadas.csv',
    'JornadaID INT, PlayaID INT, Fecha DATE, NumVoluntarios INT, TemperaturaAmbiente DOUBLE')
residuos = leer('residuos.csv',
    'RegistroID INT, JornadaID INT, TipoResiduo STRING, PesoKg DECIMAL(12,2)')

# Paso 2. Quitar las filas COMPLETAMENTE repetidas antes de validar.
# dropDuplicates() compara todas las columnas: solo elimina copias idénticas.
unicos = residuos.dropDuplicates()
recibidos = residuos.count()
duplicados = recibidos - unicos.count()
print('Recibidos:', recibidos, '| Duplicados retirados:', duplicados)
verificar('recibidos', recibidos)
verificar('duplicados', duplicados)

Recibidos: 1809 | Duplicados retirados: 5
✔ recibidos correcto
✔ duplicados correcto


In [ ]:
# TODO 1. Diagnosticar los registros defectuosos de `unicos`.
# Cada conteo debe salir de un filter() con UNA condición que escribes tú.
# 1. peso_nulo: pesajes cuyo PesoKg no tiene valor.
# 2. peso_no_positivo: pesajes cuyo peso es cero o negativo.
# 3. tipo_nulo: pesajes cuyo TipoResiduo no tiene valor.
# Usa la Referencia rápida. Recuerda: los pesos van dentro de F.col('PesoKg').
peso_nulo = unicos.filter(F.col('PesoKg').isNull()).count()
peso_no_positivo = unicos.filter(F.col('PesoKg') <= 0).count()
tipo_nulo = unicos.filter(F.col('TipoResiduo').isNull()).count()

# Esta condición YA está escrita: busca tipos que NO están en la lista válida.
tipo_fuera_lista = unicos.filter(
    ~F.col('TipoResiduo').isin('PLASTICO', 'VIDRIO', 'ORGANICO')).count()

print('Peso nulo:', peso_nulo, '| Peso <= 0:', peso_no_positivo,
      '| Tipo nulo:', tipo_nulo, '| Tipo fuera de lista:', tipo_fuera_lista)
verificar('peso_nulo', peso_nulo)
verificar('peso_no_positivo', peso_no_positivo)
verificar('tipo_nulo', tipo_nulo)
verificar('tipo_fuera_lista', tipo_fuera_lista)

Peso nulo: 1 | Peso <= 0: 1 | Tipo nulo: 1 | Tipo fuera de lista: 0
✔ peso_nulo correcto
✔ peso_no_positivo correcto
✔ tipo_nulo correcto
✔ tipo_fuera_lista correcto


**Predicción y Respuesta 1.** Hay un registro con `TipoResiduo` vacío y, sin embargo, la condición `~isin(...)` devuelve **0**. ¿Por qué no lo detecta? (Pista: en SQL, `NULL NOT IN (...)` no da verdadero ni falso.) ¿Qué condición adicional necesitarías en una validación real?

**Respuesta modelo:** Con un valor NULL, la comparación `isin` no devuelve verdadero ni falso sino desconocido (NULL), y filter solo conserva los verdaderos; por eso el registro con tipo vacío no se cuenta. Hay que añadir explícitamente la condición `isNull()` (como en la regla 'Tipo no reconocido' del Paso 4).

In [ ]:
# Paso 3. Detectar jornadas inexistentes con un LEFT JOIN.
# Renombramos JornadaID del lado de las jornadas para distinguir de qué tabla viene cada columna.
maestro = jornadas.withColumnRenamed('JornadaID', 'JornadaValida')

# 'left' conserva TODOS los pesajes. Si la jornada no existe, JornadaValida queda NULL.
# (Un 'inner' habría descartado ese pesaje en silencio.)
base = unicos.join(maestro, unicos.JornadaID == maestro.JornadaValida, 'left')

huerfanos = base.filter(F.col('JornadaValida').isNull()).count()
print('Pesajes de jornadas inexistentes:', huerfanos)
verificar('huerfanos', huerfanos)

Pesajes de jornadas inexistentes: 1
✔ huerfanos correcto


In [ ]:
# TODO 2. Asignar a cada registro UN motivo de rechazo.
# when(condición, 'motivo') se evalúa de arriba hacia abajo: gana el PRIMER motivo que se cumple.
# 1. Primer motivo: el peso no tiene valor.
# 2. Segundo motivo: el peso existe pero es cero o negativo.
# 3. Los dos motivos siguientes ya están escritos; léelos para entender el patrón.
clasificados = base.withColumn('Motivo',
    F.when(F.col('PesoKg').isNull(), 'Peso ausente')
     .when(F.col('PesoKg') <= 0, 'Peso no positivo')
     .when(F.col('TipoResiduo').isNull() |
           (~F.col('TipoResiduo').isin('PLASTICO', 'VIDRIO', 'ORGANICO')),
           'Tipo no reconocido')
     .when(F.col('JornadaValida').isNull(), 'Jornada no existente')
     .otherwise('ACEPTADO'))

validos = clasificados.filter(F.col('Motivo') == 'ACEPTADO')
rechazados = clasificados.filter(F.col('Motivo') != 'ACEPTADO')
rechazados.groupBy('Motivo').count().orderBy('Motivo').show()

aceptados, rechazos = validos.count(), rechazados.count()
verificar('motivos', {r.Motivo: r['count'] for r in rechazados.groupBy('Motivo').count().collect()})
verificar('conteos', (recibidos, duplicados, aceptados, rechazos))
tasa = round(100 * aceptados / unicos.count(), 2)
print('Aceptación entre registros únicos:', tasa, '%')
verificar('aceptacion', tasa)

+--------------------+-----+
|              Motivo|count|
+--------------------+-----+
|Jornada no existente|    1|
|        Peso ausente|    1|
|    Peso no positivo|    1|
|  Tipo no reconocido|    1|
+--------------------+-----+

✔ motivos correcto
✔ conteos correcto
Aceptación entre registros únicos: 99.78 %
✔ aceptacion correcto


In [ ]:
# Paso 4. Añadir la región SOLO a los registros aceptados y dejar la vista lista para SQL.
detalle = validos.join(playas, 'PlayaID', 'inner')
detalle.createOrReplaceTempView('residuos_limpios')   # tabla temporal para spark.sql
detalle.select('RegistroID', 'Region', 'TipoResiduo', 'PesoKg').show(5)

+----------+------+-----------+------+
|RegistroID|Region|TipoResiduo|PesoKg|
+----------+------+-----------+------+
|         2| Norte|     VIDRIO|  2.00|
|         3| Norte|   ORGANICO|  3.00|
|         7| Norte|   PLASTICO| 12.00|
|         8| Norte|     VIDRIO|  4.00|
|        10| Norte|   PLASTICO| 13.00|
+----------+------+-----------+------+
only showing top 5 rows


**Respuesta 2.** En tres líneas: 1) ¿por qué se quitan los duplicados *antes* de validar y no después? 2) ¿por qué la tasa de aceptación se calcula con 1804 y no con 1809? 3) ¿qué harías con los 4 registros rechazados: borrarlos o conservarlos? Justifica.

**Respuesta modelo:** 1) Un duplicado exacto no es un error del dato sino una repetición; si se valida antes, se cuentan dos veces los mismos defectos y se infla el número de rechazos. 2) Los 5 duplicados no son registros distintos: el universo a validar son los 1804 únicos (1800 / 1804 = 99.78 %); hay que anotar siempre el denominador. 3) Conservarlos aparte con su motivo, para que alguien corrija el origen; borrarlos esconde el problema.

## Ejercicio 04 · Consultar y explicar los indicadores (25 min)

**Qué lograremos.** Calcular toneladas por región y la variabilidad de los voluntarios por tonelada de plástico, y leer cómo Spark ejecuta una consulta.

**Idea clave: el grano.** `COUNT(*)` sobre `residuos_limpios` cuenta **pesajes** (3 por jornada), no jornadas ni voluntarios. Para estadística por jornada hay que **cambiar el grano** antes de calcular: sumar el plástico de cada jornada y recién ahí dividir.

In [ ]:
# TODO 3. Resumen por región con Spark SQL.
# Escribe cada expresión del SELECT según lo que se pide:
# 1. Registros: cuántos pesajes hay en cada región.
# 2. Kilos: el peso acumulado de cada región.
# 3. Toneladas: los kilos convertidos a toneladas (1 tonelada = 1000 kg).
# 4. Escribe la columna por la que se agrupa para obtener una fila por región.
resumen = spark.sql("""
    SELECT Region,
           COUNT(*) AS Registros,
           SUM(PesoKg) AS Kilos,
           ROUND(SUM(PesoKg) / 1000.0, 3) AS Toneladas
    FROM residuos_limpios
    GROUP BY Region
    ORDER BY Region
""")
resumen.show()
verificar('por_region', {r.Region: (r.Registros, float(r.Kilos), float(r.Toneladas))
                         for r in resumen.collect()})

+------+---------+--------+---------+
|Region|Registros|   Kilos|Toneladas|
+------+---------+--------+---------+
|Centro|      600|11998.00|   11.998|
| Norte|      600| 5998.00|    5.998|
|   Sur|      600|17998.00|   17.998|
+------+---------+--------+---------+

✔ por_region correcto


In [ ]:
# Paso 2. Cambiar el grano: de pesaje a JORNADA.
# Si sumáramos NumVoluntarios en cada fila de residuos, contaríamos los mismos voluntarios
# tres veces por jornada. Por eso primero se suma el plástico de CADA jornada.
plasticos = (detalle.filter(F.col('TipoResiduo') == 'PLASTICO')
             .groupBy('JornadaID', 'Region', 'NumVoluntarios')
             .agg(F.sum('PesoKg').alias('PlasticoKg')))     # una fila por jornada

# Tasa de cada jornada: voluntarios por tonelada de plástico (se excluyen pesos <= 0).
tasas = (plasticos.filter(F.col('PlasticoKg') > 0)
         .withColumn('VoluntariosPorTon',
                     F.col('NumVoluntarios') * 1000.0 / F.col('PlasticoKg')))
print('Jornadas con tasa calculada:', tasas.count())

Jornadas con tasa calculada: 600


In [ ]:
# TODO 4. Estadística de la tasa por región.
# 1. Jornadas: cuántas jornadas hay en cada región (ya escrito).
# 2. MediaVolPorTon: el promedio de la tasa, con dos decimales.
# 3. DesvMuestral: la desviación estándar MUESTRAL de la tasa, con dos decimales.
#    Ojo: la Referencia rápida trae la muestral y la poblacional; elige la muestral.
estadistica = (tasas.groupBy('Region').agg(
    F.count('*').alias('Jornadas'),
    F.round(F.avg('VoluntariosPorTon'), 2).alias('MediaVolPorTon'),
    F.round(F.stddev_samp('VoluntariosPorTon'), 2).alias('DesvMuestral')))
estadistica.orderBy('Region').show()
verificar('estadistica', {r.Region: (r.Jornadas, r.MediaVolPorTon, r.DesvMuestral)
                          for r in estadistica.collect()})

+------+--------+--------------+------------+
|Region|Jornadas|MediaVolPorTon|DesvMuestral|
+------+--------+--------------+------------+
|Centro|     200|        398.08|       51.38|
| Norte|     200|         749.9|      213.39|
|   Sur|     200|        295.63|       26.86|
+------+--------+--------------+------------+

✔ estadistica correcto


In [ ]:
# TODO 5. Clasificar cada jornada por el peso total recolectado.
# 1. Primero se suma el peso de TODOS los residuos de cada jornada (ya escrito).
# 2. Escribe la condición: la jornada es de Alto Impacto cuando sus Kilos llegan a 60 o más.
#    (El umbral de 60 kg es didáctico; no es una norma ambiental.)
impacto = (detalle.groupBy('JornadaID').agg(F.sum('PesoKg').alias('Kilos'))
           .withColumn('Impacto',
                       F.when(F.col('Kilos') >= 60, 'Alto Impacto').otherwise('Bajo Impacto')))
impacto.groupBy('Impacto').count().orderBy('Impacto').show()
verificar('impacto', {r.Impacto: r['count'] for r in impacto.groupBy('Impacto').count().collect()})

+------------+-----+
|     Impacto|count|
+------------+-----+
|Alto Impacto|  303|
|Bajo Impacto|  297|
+------------+-----+

✔ impacto correcto


In [ ]:
# TODO 6. Leer el plan de ejecución de Spark.
# 1. explain() muestra CÓMO ejecutará Spark una consulta (todavía sin ejecutarla).
# 2. Aquí usamos una consulta pequeña: voluntarios acumulados por playa.
# 3. Léelo de ABAJO hacia ARRIBA. Busca estas palabras y anota qué hace cada una:
#    FileScan csv  ·  HashAggregate  ·  Exchange
# 4. No hay nada que completar: la explicación va en la Respuesta 3.
voluntarios = jornadas.groupBy('PlayaID').agg(F.sum('NumVoluntarios').alias('Voluntarios'))
voluntarios.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[PlayaID#10], functions=[sum(NumVoluntarios#12)])
   +- Exchange hashpartitioning(PlayaID#10, 2), ENSURE_REQUIREMENTS, [plan_id=4447]
      +- HashAggregate(keys=[PlayaID#10], functions=[partial_sum(NumVoluntarios#12)])
         +- FileScan csv [PlayaID#10,NumVoluntarios#12] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/claude/datos/jornadas.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<PlayaID:int,NumVoluntarios:int>


**Respuesta 3.** En cuatro líneas: 1) ¿por qué no debemos sumar `NumVoluntarios` directamente en cada fila de residuos? 2) ¿qué representa `DesvMuestral` y qué región tiene jornadas más desiguales? 3) ¿por qué usamos la desviación muestral y no la poblacional? 4) según el plan de `explain()`, ¿qué hace `Exchange`?

**Respuesta modelo:** 1) Cada jornada tiene tres pesajes; sumar voluntarios por fila contaría tres veces los mismos voluntarios y daría una cifra equivocada. 2) La dispersión de la tasa de las jornadas alrededor de su media; Norte (213.39) es la región con jornadas más desiguales. 3) Las jornadas son una muestra del funcionamiento de la ONG, no la población completa; la muestral divide entre n−1. 4) Exchange es la redistribución (shuffle) de datos entre particiones para juntar las claves iguales antes de agregar.

## Ejercicio 05 · Comunicar y conservar los resultados (25 min)

**Qué lograremos.** Guardar el detalle limpio en **Parquet** (formato de columnas que conserva los tipos) y llevar un resumen de tres filas a **Power BI Desktop** para comunicar un hallazgo.

**Regla de oro.** Al Power BI y a los correos viajan **resúmenes pequeños**; el detalle masivo se queda en Parquet. Un `toPandas()` sobre millones de filas agota la memoria.

In [ ]:
# Paso 1. Guardar y releer el detalle limpio en Parquet.
from pathlib import Path
import shutil
SALIDA = Path.cwd() / 'salidas_tema07'
SALIDA.mkdir(exist_ok=True)

# Spark no escribe UN archivo: escribe una CARPETA con varios archivos de datos.
detalle.write.mode('overwrite').parquet(str(SALIDA / 'detalle_parquet'))
releido = spark.read.parquet(str(SALIDA / 'detalle_parquet'))
print('Filas releídas:', releido.count())
verificar('parquet', releido.count())

Filas releídas: 1800
✔ parquet correcto


In [ ]:
# TODO 7. Exportar el resumen de tres filas para Power BI.
# 1. panel junta el resumen por región con la estadística (3 filas en total).
# 2. Convierte panel a un DataFrame de pandas con el método de la Referencia rápida
#    que trae TODO al driver (aquí es seguro: son 3 filas).
# 3. .to_csv guarda el archivo; utf-8-sig evita problemas con las tildes en Power BI.
panel = resumen.join(estadistica, 'Region').orderBy('Region')
archivo = SALIDA / 'resumen_region.csv'
panel.toPandas().to_csv(archivo, index=False, encoding='utf-8-sig')

# Empaquetar CSV y Parquet para entregarlos (crea resultados_tema07.zip en la carpeta actual).
paquete = shutil.make_archive('resultados_tema07', 'zip', SALIDA)
print('Archivos listos:', archivo.name, '|', Path(paquete).name)
verificar('csv_filas', len(open(archivo, encoding='utf-8-sig').read().strip().splitlines()) - 1)

Archivos listos: resumen_region.csv | resultados_tema07.zip
✔ csv_filas correcto


In [ ]:
# Paso 2. Descargar antes de cerrar la sesión de Colab.
from google.colab import files
files.download(str(archivo))
files.download(paquete)

### Construir el reporte en Power BI Desktop
1. *Obtener datos > Texto/CSV* > `resumen_region.csv` > *Transformar datos*.
2. Comprueba los tipos: `Region` texto; `Registros` y `Jornadas` enteros; `Kilos`, `Toneladas`, `MediaVolPorTon` y `DesvMuestral` número decimal. Si el punto decimal se interpreta mal: *Tipo de datos > Usar configuración regional > Inglés (Estados Unidos)*.
3. *Cerrar y aplicar*. Inserta un **gráfico de barras**: `Region` en el eje y `Toneladas` en valores, orden descendente.
4. Título del gráfico: *¿En qué región se recolectó más peso de residuos?* Indica la unidad (Toneladas) en el eje.
5. Agrega una **tarjeta** con la suma de `Toneladas` y una **tabla** con `Region`, `MediaVolPorTon` y `DesvMuestral` (sin resumir; desactiva el total).
6. Comprueba el **total sin filtros**: debe ser **35.994** toneladas. Guarda `OceanCleanUp_Tema07.pbix` y una captura.

In [ ]:
# Control de la tarjeta de Power BI: compara con el total de Spark.
total_t = spark.sql('SELECT ROUND(SUM(PesoKg) / 1000.0, 3) FROM residuos_limpios').first()[0]
print('Toneladas totales según Spark:', total_t)
verificar('toneladas_total', float(total_t))

Toneladas totales según Spark: 35.994
✔ toneladas_total correcto


**Respuesta 4 · TODO 8 (comunicar con límites).** Escribe en tres líneas, con esta estructura:
1. **Decisión** que el gráfico permite apoyar.
2. **Límite:** una conclusión que *no* se puede deducir del peso recolectado (considera el esfuerzo de limpieza y la cobertura de las jornadas).
3. **Dato que pedirías** para cerrar ese límite.

**Respuesta modelo:** 1) Decidir en qué región priorizar el transporte y la logística de recolección (Sur concentra más peso: 17.998 t). 2) No se puede concluir que Sur esté más contaminada: el peso recolectado depende del esfuerzo (voluntarios, jornadas) y de la cobertura, no del nivel de contaminación; Norte puede tener menos peso por menos esfuerzo. 3) Pediría un indicador de contaminación independiente (por ejemplo el nivel de los sensores) o el esfuerzo por jornada para normalizar el peso.

**Pista:** peso recolectado e índice de contaminación representan variables diferentes. Usa unidades en títulos y ejes.

### Reto integrador (opcional, fuera de los 90 minutos)
**Resumen de plástico por región.** La coordinación solicita un resumen exclusivo de `PLASTICO` para planificar su transporte.

1. Lee de nuevo `salidas_tema07/detalle_parquet` en un DataFrame llamado `base_reto` y comprueba que conserva 1800 filas.
2. Registra la vista temporal `reto_residuos` y escribe una consulta Spark SQL que conserve solo `TipoResiduo = 'PLASTICO'` y devuelva una fila por `Region`.
3. Muestra `Region`, `Pesajes`, `ToneladasPlastico` (3 decimales) y `PromedioKgPorPesaje` (2 decimales), ordenado por toneladas de mayor a menor.
4. Comprueba que hay tres regiones y que sus pesajes suman 600. Exporta como `reto_plastico.csv`.
5. Responde en dos líneas: qué región concentra más plástico y por qué ese total no se compara con las 35.994 toneladas de todos los residuos.

*Control:* 1800 filas de entrada; tres filas de salida; 600 pesajes de plástico. El filtro de tipo va **antes** de agrupar.

In [ ]:
# Reto. Escribe aquí tu solución completa.
# Reutiliza lo practicado en E03, E04 y E05. No modifiques los archivos originales.
base_reto = spark.read.parquet(str(SALIDA / 'detalle_parquet'))
print('Filas:', base_reto.count())
base_reto.createOrReplaceTempView('reto_residuos')
reto = spark.sql("""
    SELECT Region,
           COUNT(*) AS Pesajes,
           ROUND(SUM(PesoKg) / 1000.0, 3) AS ToneladasPlastico,
           ROUND(AVG(PesoKg), 2) AS PromedioKgPorPesaje
    FROM reto_residuos
    WHERE TipoResiduo = 'PLASTICO'
    GROUP BY Region
    ORDER BY ToneladasPlastico DESC
""")
reto.show()
print('Suma de pesajes:', sum(r.Pesajes for r in reto.collect()))
reto.toPandas().to_csv('reto_plastico.csv', index=False, encoding='utf-8-sig')

Filas: 1800
+------+-------+-----------------+-------------------+
|Region|Pesajes|ToneladasPlastico|PromedioKgPorPesaje|
+------+-------+-----------------+-------------------+
|   Sur|    200|           11.900|              59.50|
|Centro|    200|            7.900|              39.50|
| Norte|    200|            3.900|              19.50|
+------+-------+-----------------+-------------------+

Suma de pesajes: 600


## Preguntas de cierre (responde en una línea cada una)
1. ¿Qué diferencia hay entre un duplicado exacto y un registro que incumple una regla?
2. ¿Por qué `COUNT(*)` en `residuos_limpios` no sirve para contar jornadas?
3. ¿Qué pasaría si usaras un `inner join` en lugar de `left` en el Paso 3 de E03?
4. ¿Por qué se envía a Power BI un CSV de 3 filas y no el detalle?
5. ¿Qué tabla de **tu desafío** necesitaría las mismas reglas de calidad y cuáles serían?

**Respuesta modelo:** 1) El duplicado es la misma fila repetida (no aporta información nueva); el registro que incumple una regla es un dato distinto pero inválido (peso ausente, negativo, tipo o jornada inexistentes). 2) Cada jornada tiene varios pesajes; COUNT(*) cuenta pesajes (grano pesaje), no jornadas. 3) El pesaje de la jornada inexistente desaparecería en silencio y no podría reportarse como rechazo. 4) Es un resumen: el detalle masivo no cabe en memoria ni hace falta para el gráfico; así se evita toPandas() sobre millones de filas. 5) Respuesta abierta: debe identificar una tabla de hechos de su desafío y reglas de nulos, duplicados, valores no positivos y claves sin correspondencia.